# Notebook 3 — Classic Convolutional Architectures: ResNet vs VGG

## Story
Deeper networks help, but simply stacking more layers causes the **vanishing
gradient problem** — gradients shrink to near-zero before reaching early layers.

Two landmark architectures solved this differently:

| Architecture | Key Idea |
|---|---|
| **VGG** (2014) | Very deep networks of small (3×3) conv filters stacked in blocks. No skip connections. Simple but large. |
| **ResNet** (2015) | Residual (skip) connections: the output of a block is `F(x) + x`, letting gradients bypass any block. |

We build simplified versions of **both** architectures from scratch (no pretrained
weights) and compare them on our task.

**Labels (12):** `pen, paper, book, clock, phone, laptop, chair, desk, bottle, keychain, backpack, calculator`

## 1. Imports & Setup

In [ ]:
import sys
sys.path.insert(0, "../")
sys.path.insert(0, "../experiments")

import torch
import torch.nn as nn
from pathlib import Path

from eval import LABEL_ORDER
from utils import (
    set_seed, load_dataset, split_dataset, subsample_subset,
    get_train_transform, get_eval_transform, build_dataloaders,
    run_baselines, print_model_info,
    train_model, save_checkpoint, load_checkpoint, plot_training_history,
    plot_multi_arch_histories,
    collect_test_predictions, categorize_predictions,
    show_prediction_examples, plot_per_class_metrics,
    plot_confusion_matrices, show_saliency_examples,
    compute_multilabel_metrics, NUM_LABELS, METRIC_KEYS,
)

SEED   = 42
set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## 2. Config

In [ ]:
BASE_DATA_DIR   = "../data/aggregated"
IMAGE_SIZE      = 128
BATCH_SIZE      = 256
SPLIT           = [0.8, 0.1, 0.1]
USE_SUBSET      = False
SUBSET_FRACTION = 0.1
CHECKPOINT_DIR  = Path("../checkpoints")
print(f"Labels ({NUM_LABELS}): {LABEL_ORDER}")

## 3. Data Loading & Loaders

In [ ]:
full_dataset = load_dataset(BASE_DATA_DIR)
train_raw, val_raw, test_raw = split_dataset(full_dataset, SPLIT, SEED)

if USE_SUBSET:
    train_raw = subsample_subset(train_raw, SUBSET_FRACTION, SEED)
    val_raw   = subsample_subset(val_raw,   SUBSET_FRACTION, SEED + 1)
    test_raw  = subsample_subset(test_raw,  SUBSET_FRACTION, SEED + 2)

train_transform = get_train_transform(IMAGE_SIZE)
eval_transform  = get_eval_transform(IMAGE_SIZE)
train_loader, val_loader, test_loader = build_dataloaders(
    train_raw, val_raw, test_raw, train_transform, eval_transform, BATCH_SIZE
)
print(f"Train: {len(train_raw)}  Val: {len(val_raw)}  Test: {len(test_raw)}")

## 4. Architecture A — Small ResNet (with Residual Connections)

```
Stem:  Conv(3→32) + BN + ReLU
Layer1: ResBlock(32→64,  stride=2)  →  64×64
Layer2: ResBlock(64→128, stride=2)  →  32×32
Layer3: ResBlock(128→256,stride=2)  →  16×16
GAP → Dropout(0.4) → Linear(256→12)
```
The `ResBlock` adds the input back to the output via a 1×1 skip convolution.

In [ ]:
class ResidualBlock(nn.Module):
    """Conv → BN → ReLU → Conv → BN  +  1×1 skip projection."""

    def __init__(self, in_ch: int, out_ch: int, stride: int = 1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch,  out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.relu  = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, stride=1,      padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)
        self.skip  = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.skip = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + self.skip(x))


class SmallResNet(nn.Module):
    def __init__(self, num_labels: int):
        super().__init__()
        self.stem   = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
        )
        self.layer1 = ResidualBlock(32,  64,  stride=2)
        self.layer2 = ResidualBlock(64,  128, stride=2)
        self.layer3 = ResidualBlock(128, 256, stride=2)
        self.gap    = nn.AdaptiveAvgPool2d(1)
        self.head   = nn.Sequential(nn.Dropout(p=0.4), nn.Linear(256, num_labels))

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        return self.head(torch.flatten(self.gap(x), 1))


def create_small_resnet(num_labels: int) -> nn.Module:
    return SmallResNet(num_labels)


print_model_info(create_small_resnet(NUM_LABELS).to(DEVICE))

## 5. Architecture B — VGG-Style Net (no skip connections)

Classic VGG stacks multiple 3×3 convolutions within each block before
downsampling. There are **no skip connections** — the network must learn
everything through the main path.

```
Block 1:  2× Conv(3→64,   3×3) + BN + ReLU → MaxPool  →  64×64
Block 2:  2× Conv(64→128, 3×3) + BN + ReLU → MaxPool  →  32×32
Block 3:  3× Conv(128→256,3×3) + BN + ReLU → MaxPool  →  16×16
GAP → Dropout(0.5) → Linear(256→256) → ReLU → Dropout → Linear(256→12)
```

In [ ]:
class VGGStyleNet(nn.Module):
    """VGG-inspired architecture: stacked 3×3 convs per block, no residual connections."""

    def __init__(self, num_labels: int):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1: 128×128 → 64×64
            nn.Conv2d(3,   64,  3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(inplace=True),
            nn.Conv2d(64,  64,  3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            # Block 2: 64×64 → 32×32
            nn.Conv2d(64,  128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            # Block 3: 32×32 → 16×16  (3 conv layers, VGG-D style)
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.gap  = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(256, 256), nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_labels),
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x).flatten(1)
        return self.head(x)


def create_vgg_style(num_labels: int) -> nn.Module:
    return VGGStyleNet(num_labels)


print_model_info(create_vgg_style(NUM_LABELS).to(DEVICE))

## 6. Train SmallResNet

In [ ]:
RESNET_PATH = CHECKPOINT_DIR / "small_resnet.pth"

best_state_resnet, best_f1_resnet, history_resnet, epochs_resnet = train_model(
    create_small_resnet, NUM_LABELS, train_loader, val_loader, DEVICE,
    lr=1e-3, weight_decay=1e-3, max_epochs=40,
)
save_checkpoint(best_state_resnet, RESNET_PATH)
print(f"\nSmallResNet best val F1: {best_f1_resnet:.4f}")
plot_training_history(history_resnet, epochs_resnet, "SmallResNet", 1e-3, 1e-3)

## 7. Train VGGStyleNet

In [ ]:
VGG_PATH = CHECKPOINT_DIR / "vgg_style.pth"

best_state_vgg, best_f1_vgg, history_vgg, epochs_vgg = train_model(
    create_vgg_style, NUM_LABELS, train_loader, val_loader, DEVICE,
    lr=1e-3, weight_decay=1e-3, max_epochs=40,
)
save_checkpoint(best_state_vgg, VGG_PATH)
print(f"\nVGGStyleNet best val F1: {best_f1_vgg:.4f}")
plot_training_history(history_vgg, epochs_vgg, "VGGStyleNet", 1e-3, 1e-3)

## 8. Compare Architectures — Training Curves

In [ ]:
all_histories = {"SmallResNet": history_resnet, "VGGStyleNet": history_vgg}
plot_multi_arch_histories(all_histories, experiment_name="Classic CNNs (from scratch)")

## 9. Test-Set Metrics Comparison

In [ ]:
models_to_eval = {
    "SmallResNet": (create_small_resnet, RESNET_PATH),
    "VGGStyleNet": (create_vgg_style,    VGG_PATH),
}

print(f"{'Model':<14} {'loss':>7} {'exact':>7} {'hamming':>8} {'IoU':>7} {'prec':>7} {'rec':>7} {'F1':>7}")
print("-" * 75)
for name, (fn, ckpt) in models_to_eval.items():
    m = load_checkpoint(fn, NUM_LABELS, ckpt, DEVICE)
    imgs, lbls, preds, probs = collect_test_predictions(m, test_loader, DEVICE)
    all_logits = []
    m.eval()
    with torch.no_grad():
        for images, _ in test_loader:
            all_logits.append(m(images.to(DEVICE)).cpu())
    logits = torch.cat(all_logits)
    met = compute_multilabel_metrics(lbls, preds, logits=logits)
    print(f"{name:<14} {met['loss']:>7.4f} {met['exact_match']:>7.4f} "
          f"{met['hamming_acc']:>8.4f} {met['mean_iou']:>7.4f} "
          f"{met['precision_micro']:>7.4f} {met['recall_micro']:>7.4f} "
          f"{met['f1_micro']:>7.4f}")

## 10. Analysis — Best Classic CNN

In [ ]:
# Analyze the SmallResNet (typically better due to skip connections)
model = load_checkpoint(create_small_resnet, NUM_LABELS, RESNET_PATH, DEVICE)
images, labels, preds, probs = collect_test_predictions(model, test_loader, DEVICE)
correct_idx, partial_idx, incorrect_idx = categorize_predictions(labels, preds)

show_prediction_examples(correct_idx,   images, labels, preds, "Fully Correct",     n=4)
show_prediction_examples(partial_idx,   images, labels, preds, "Partially Correct", n=4)
show_prediction_examples(incorrect_idx, images, labels, preds, "Fully Incorrect",   n=4)

In [ ]:
plot_per_class_metrics(labels, preds)
plot_confusion_matrices(labels, preds)

In [ ]:
show_saliency_examples(correct_idx,   images, labels, preds, model, "Fully Correct",   n=4, device=DEVICE)
show_saliency_examples(incorrect_idx, images, labels, preds, model, "Fully Incorrect", n=4, device=DEVICE)

## 11. Conclusions

- Both ResNet and VGG-style networks outperform the flat CNNs from Notebook 2.
- **Skip connections (ResNet) generally outperform plain stacks (VGG)** —
  gradients flow better through residual paths.
- Yet both are still *trained from scratch*, so they can only exploit the
  features learnable from our dataset alone.
- The performance ceiling has not been reached — we need **pretrained weights**.

**Next:** We leverage ImageNet-pretrained lightweight models — MobileNet and
EfficientNet — to see how much pretrained features help (Notebook 4).